In [ ]:
"""
Sample Script: transform_clean.py
Confidentiality Notice: This script is a sample representation and contains no actual client data.
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, when

In [ ]:
# Initialize Spark
spark = SparkSession.builder \
    .appName("TransformCleanSAPData") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

In [ ]:
# Read from Bronze Delta layer
df = spark.read.format("delta").load("dbfs:/mnt/bronze_layer/sample_data")

# Optimization: Cache the data if reused
df.cache()

In [ ]:
# Transformation Logic
df_clean = (
    df.dropDuplicates(["id"])  # Avoid shuffling by specifying key
      .filter(col("status").isNotNull())
      .withColumn("region", trim(col("region")))
      .withColumn("amount", when(col("amount").isNull(), 0).otherwise(col("amount")))
)

In [ ]:
# Optimization: Repartition if writing large data
df_clean = df_clean.repartition("region")

In [ ]:
# Write to Silver Delta layer (partitioned)
df_clean.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("region") \
    .save("dbfs:/mnt/silver_layer/sample_data")

spark.stop()